In [3]:
# Install the required Python packages
!pip install gradio kokoro soundfile torch numpy

# Kokoro requires 'espeak-ng' as a system dependency to read text properly:
# On Windows: Download the installer from https://github.com/espeak-ng/espeak-ng/releases
# On Mac: brew install espeak
# On Linux: sudo apt-get install espeak-ng

In [5]:
import gradio as gr
import torch
import os
import glob
import numpy as np
import urllib.request

# 1. Automated Voice Setup
def ensure_voices_exist():
    """Checks for existing voices and downloads a starter pack if the folder is empty."""
    os.makedirs("voices", exist_ok=True)
    existing_voices = glob.glob("voices/*.pt")

    if len(existing_voices) > 0:
        return # Voices already exist, no need to download

    print("\n[!] No voices found in 'voices/' directory.")
    print("[+] Downloading a starter pack of 10 popular Kokoro voices from Hugging Face...")

    starter_pack = [
        "af_bella", "af_sarah", "af_nicole", "af_sky",
        "am_adam", "am_michael", "bf_emma", "bf_isabella",
        "bm_george", "bm_lewis"
    ]

    base_url = "https://huggingface.co/hexgrad/Kokoro-82M/resolve/main/voices/{}.pt"

    for voice_name in starter_pack:
        file_path = os.path.join("voices", f"{voice_name}.pt")
        url = base_url.format(voice_name)
        try:
            print(f"    -> Downloading {voice_name}.pt...")
            urllib.request.urlretrieve(url, file_path)
        except Exception as e:
            print(f"    -> Failed to download {voice_name}: {e}")

    print("[+] Starter pack downloaded successfully!\n")

# Run the voice check before we do anything else
ensure_voices_exist()

# Fetch the available voices for the dropdowns
AVAILABLE_VOICES = [
    os.path.basename(f)[:-3]
    for f in glob.glob("voices/*.pt")
    if not os.path.basename(f).startswith("my_custom") # Hide previous custom blends from crowding the list
]
AVAILABLE_VOICES.sort() # Alphabetize the dropdown list

# 2. Initialize the Kokoro Pipeline
try:
    from kokoro import KPipeline
    print("Loading Kokoro model...")
    pipeline = KPipeline(lang_code='a')
    print("Model loaded successfully!")
except ImportError:
    pipeline = None
    print("Error: 'kokoro' library not found. Please pip install kokoro")

# 3. Blending and Generation Logic
def blend_and_test(v1_name, w1, v2_name, w2, v3_name, w3, sample_text):
    """Blends the selected voices, passes the tensor to Kokoro in-memory, and saves a file for download."""
    if pipeline is None:
        raise gr.Error("Kokoro pipeline failed to load. Please ensure it is installed.")

    selections = [(v1_name, w1), (v2_name, w2), (v3_name, w3)]

    # Filter out empty dropdowns or 0% weights
    valid = [(v, w) for v, w in selections if v and w > 0]

    if not valid:
        raise gr.Error("Please select at least one valid voice and set its weight above 0%.")

    total_weight = sum(w for _, w in valid)

    # Mathematically blend the tensors
    blended_tensor = None
    for name, weight in valid:
        path = os.path.join("voices", f"{name}.pt")
        tensor = torch.load(path, weights_only=True)
        normalized_w = weight / total_weight # Ensure weights always equal exactly 100%

        if blended_tensor is None:
            blended_tensor = tensor * normalized_w
        else:
            blended_tensor += tensor * normalized_w

    # Save the blended tensor to disk ONLY so the user can download it via the Gradio File UI
    output_name = "my_custom_blend.pt"
    output_path = os.path.join("voices", output_name)
    torch.save(blended_tensor, output_path)

    # Generate the audio using Kokoro
    try:
        # THE FIX: We pass the tensor object directly to `voice=` instead of a string name.
        # This prevents the library from pinging HuggingFace for a 404 URL.
        generator = pipeline(sample_text, voice=blended_tensor, speed=1.0)
        audio_chunks = []

        for i, (gs, ps, audio) in enumerate(generator):
            if audio is not None:
                audio_chunks.append(audio)

        if not audio_chunks:
            raise gr.Error("Audio generation resulted in empty output.")

        final_audio = np.concatenate(audio_chunks)

    except Exception as e:
        raise gr.Error(f"Generation failed: {str(e)}")

    # Return audio tuple for playback, and the file path for download
    return (24000, final_audio), output_path

# 4. Build the User Interface
with gr.Blocks(theme=gr.themes.Soft(), title="Kokoro Voice Blender") as app:
    gr.Markdown("# 🎛️ Kokoro Custom Voice Blender")
    gr.Markdown("Select up to 3 voices, adjust their weights, and generate a sample. The final `.pt` file will be available to download.")

    with gr.Row():
        # Left Column: The Levers
        with gr.Column():
            gr.Markdown("### 1. Blend Settings")

            with gr.Group():
                with gr.Row():
                    v1 = gr.Dropdown(choices=AVAILABLE_VOICES, label="Voice 1", value=AVAILABLE_VOICES[0] if AVAILABLE_VOICES else None)
                    w1 = gr.Slider(0, 100, value=50, label="Weight %")

                with gr.Row():
                    v2 = gr.Dropdown(choices=AVAILABLE_VOICES, label="Voice 2", value=AVAILABLE_VOICES[1] if len(AVAILABLE_VOICES)>1 else None)
                    w2 = gr.Slider(0, 100, value=50, label="Weight %")

                with gr.Row():
                    v3 = gr.Dropdown(choices=AVAILABLE_VOICES, label="Voice 3 (Optional)")
                    w3 = gr.Slider(0, 100, value=0, label="Weight %")

            gr.Markdown("### 2. Test Prompt")
            text_input = gr.Textbox(
                label="Sample Text",
                value="Hello! I am a brand new custom voice created by mathematically blending different profiles together. How do I sound?",
                lines=3
            )

            btn = gr.Button("Blend & Generate Sample 🎙️", variant="primary", size="lg")

        # Right Column: The Output
        with gr.Column():
            gr.Markdown("### 3. Output")
            audio_out = gr.Audio(label="Preview Audio", autoplay=True)

            gr.Markdown("### 4. Save Your Voice")
            gr.Markdown("*If you like how it sounds, download the `.pt` file below and put it in your production Kokoro `voices` folder.*")
            file_out = gr.File(label="Download Custom Voice (.pt)")

    # Connect the button to the generation function
    btn.click(
        fn=blend_and_test,
        inputs=[v1, w1, v2, w2, v3, w3, text_input],
        outputs=[audio_out, file_out]
    )

if __name__ == "__main__":
    print("\nStarting Web UI...")
    app.launch(inbrowser=True)

Loading Kokoro model...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model loaded successfully!


/tmp/ipykernel_1876/4060044118.py:116: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Kokoro Voice Blender") as app:



Starting Web UI...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d04fe7b05fb5f427e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
